# Graph similarity via GIN embeddings + cosine similarity

This reuses the GIN message-passing layer and Jumping-Knowledge sum readout from [GGIN.ipynb](GGIN.ipynb), but instead of *classifying* a graph we stop at the **graph embedding**: each graph is encoded into a single fixed-length vector living in the same space, and the similarity of two graphs is the **cosine of the angle** between their vectors.

**Why this works even without training.** GIN is permutation invariant, so two isomorphic graphs map to the *exact* same vector $\Rightarrow$ cosine similarity $1.0$, regardless of node labelling and regardless of the (random) weights. A small structural change moves the vector a little; a large one moves it a lot. So a randomly-initialised GIN already behaves like a structural graph kernel. (You could train it for a task-specific notion of similarity, but the structural signal is there out of the box.)

**Guaranteed $[0, 1]$ output.** Every layer ends in ReLU and the readout is a sum over nodes, so every embedding lives in the non-negative orthant. The cosine of two non-negative vectors is always in $[0, 1]$: $1.0$ = identical direction (isomorphic / very similar), values toward $0$ = structurally dissimilar.

## 1. The GIN encoder

The `GINLayer` is the message passing from GGIN (Eq. 4.1):

$$h_v^{(k)} = \text{MLP}^{(k)}\!\Big( (1 + \epsilon)\cdot h_v^{(k-1)} + \sum_{u \in \mathcal{N}(v)} h_u^{(k-1)} \Big)$$

The `GINEncoder` stacks $K$ layers and, instead of a classification head, applies the JK readout (Eq. 4.2): sum node features within each layer (including the raw input $h^{(0)}$) and concatenate across layers into one graph vector. ReLU on every MLP output keeps features non-negative, which is what pins the final cosine into $[0, 1]$.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducible random weights -> the same embeddings/scores every run.
torch.manual_seed(42)


class GINLayer(nn.Module):
    """One GIN message-passing layer (Eq. 4.1 of Xu et al., 2019).

    h_v = MLP( (1 + eps) * h_v + sum_{u in N(v)} h_u ).  The sum aggregator is
    injective over multisets, which is what makes GIN as strong as the 1-WL test.
    ReLU on the MLP output keeps features non-negative (needed for a [0,1] cosine).
    """

    def __init__(self, in_dim, out_dim, eps=0.0):
        super().__init__()
        self.eps = eps
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim),
            nn.ReLU(),
        )

    def forward(self, A, X):
        # A @ X is the SUM of each node's neighbour features (A is the adjacency).
        neighbor_sum = A @ X
        out = (1.0 + self.eps) * X + neighbor_sum
        return self.mlp(out)


class GINEncoder(nn.Module):
    """Encode a graph into a single vector via GIN layers + JK sum readout.

    The readout concatenates, across every layer (including the raw input h^(0)),
    the sum of node features. Early layers carry local structure, later layers more
    global structure, so concatenating keeps all of it (Eq. 4.2).
    """

    def __init__(self, in_dim, hidden_dim=32, num_layers=5):
        super().__init__()
        self.layers = nn.ModuleList()
        self.layers.append(GINLayer(in_dim, hidden_dim))
        for _ in range(num_layers - 1):
            self.layers.append(GINLayer(hidden_dim, hidden_dim))
        self.out_dim = in_dim + num_layers * hidden_dim

    def forward(self, A, X):
        reps = [X]                      # h^(0) = raw node features
        h = X
        for layer in self.layers:
            h = layer(A, h)
            reps.append(h)              # h^(k)
        # READOUT = sum over nodes per layer, then CONCAT across layers.
        pooled = [h_k.sum(dim=0) for h_k in reps]
        return torch.cat(pooled, dim=-1)   # (out_dim,)


def cosine_similarity(vec_a, vec_b):
    """Cosine similarity of two graph embeddings, clamped to [0, 1]."""
    sim = F.cosine_similarity(vec_a, vec_b, dim=0)
    return float(sim.clamp(0.0, 1.0))

## 2. Turning an edge list into (adjacency `A`, node features `X`)

Each graph is built from an undirected edge list. The node feature is the **one-hot encoding of the node's degree** — a permutation-equivariant, label-free structural feature, so we compare pure *shape* rather than node names.

In [3]:
FEATURE_DIM = 8   # one-hot node-degree buckets 0..7 (degrees >=7 are clamped)


class Graph:
    """A tiny undirected graph built from an edge list."""

    def __init__(self, num_nodes, edges, name=""):
        self.name = name
        self.num_nodes = num_nodes
        self.edges = edges
        self.A = self._adjacency(num_nodes, edges)
        self.X = self._degree_features(self.A)

    @staticmethod
    def _adjacency(num_nodes, edges):
        A = torch.zeros((num_nodes, num_nodes))
        for u, v in edges:
            A[u, v] = 1.0
            A[v, u] = 1.0 
        return A

    @staticmethod
    def _degree_features(A):
        """Node feature = one-hot of the node's degree (a permutation-equivariant,
        label-free structural feature so we compare pure shape, not node names)."""
        degrees = A.sum(dim=1).long().clamp(max=FEATURE_DIM - 1)
        return F.one_hot(degrees, num_classes=FEATURE_DIM).float()

    def embed(self, encoder):
        return encoder(self.A, self.X)


# Handy constructors for the demo graphs.
def cycle(n, name=None):
    edges = [(i, (i + 1) % n) for i in range(n)]
    return Graph(n, edges, name or f"C{n} (cycle-{n})")


def path(n, name=None):
    edges = [(i, i + 1) for i in range(n - 1)]
    return Graph(n, edges, name or f"P{n} (path-{n})")


def star(n_leaves, name=None):
    edges = [(0, i) for i in range(1, n_leaves + 1)]
    return Graph(n_leaves + 1, edges, name or f"S{n_leaves} (star, {n_leaves} leaves)")


def complete(n, name=None):
    edges = [(i, j) for i in range(n) for j in range(i + 1, n)]
    return Graph(n, edges, name or f"K{n} (complete-{n})")

## 3. Build the encoder and the demo graphs

We instantiate the encoder once (fixed random weights $\Rightarrow$ a deterministic structural encoder) and define the graphs we'll compare: two triangles, two 6-cycles, a path and a small perturbation of it, a star, and the classic *two-triangles vs 6-cycle* pair.

In [4]:
encoder = GINEncoder(in_dim=FEATURE_DIM, hidden_dim=32, num_layers=3)
encoder.eval()   # fixed random weights -> deterministic structural encoder


def score(g1, g2):
    """Cosine similarity of two graphs' GIN embeddings."""
    with torch.no_grad():
        return cosine_similarity(g1.embed(encoder), g2.embed(encoder))


print(f"GIN graph encoder -> {encoder.out_dim}-dim embeddings, "
      f"cosine similarity in [0, 1]")

# --- The graphs ---
triangle_a = cycle(3, "triangle A  (0-1-2)")
# Same triangle, nodes RELABELLED -> must be isomorphic to triangle_a.
triangle_b = Graph(3, [(2, 0), (0, 1), (1, 2)], "triangle B  (relabelled)")

c6_a = cycle(6, "6-cycle A")
c6_b = Graph(6, [(5, 0), (0, 1), (1, 2), (2, 3), (3, 4), (4, 5)], "6-cycle B (relabelled)")

two_triangles = Graph(6, [(0, 1), (1, 2), (2, 0), (3, 4), (4, 5), (5, 3)],
                      "two triangles (2 x C3)")

p6 = path(6, "path P6")
# Small perturbation of P6: add ONE edge (0-5) -> turns the path into a 6-cycle.
p6_plus_edge = Graph(6, [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (0, 5)],
                     "P6 + 1 edge  (small perturbation)")

s5 = star(5, "star S5")

GIN graph encoder -> 104-dim embeddings, cosine similarity in [0, 1]


## 4. Test cases

In [7]:
cases = [
    ("isomorphic (relabelled), expect ~1.00", triangle_a, triangle_b),
    ("isomorphic (relabelled), expect ~1.00", c6_a, c6_b),
    ("+1 edge, expect high but < 1.00",        p6, p6_plus_edge),
    ("dissimilar, expect low",                 p6, s5),
    ("dissimilar, expect low",                 triangle_a, s5),
    ("1-WL / GIN blind spot",                  two_triangles, c6_a),
]

print(f"{'case':<40}{'graph A':<26}{'graph B':<26}{'cosine':>8}")
print("-" * 100)
for label, g1, g2 in cases:
    print(f"{label:<40}{g1.name:<26}{g2.name:<26}{score(g1, g2):>8.4f}")

case                                    graph A                   graph B                     cosine
----------------------------------------------------------------------------------------------------
isomorphic (relabelled), expect ~1.00   triangle A  (0-1-2)       triangle B  (relabelled)    1.0000
isomorphic (relabelled), expect ~1.00   6-cycle A                 6-cycle B (relabelled)      1.0000
+1 edge, expect high but < 1.00         path P6                   P6 + 1 edge  (small perturbation)  0.9296
dissimilar, expect low                  path P6                   star S5                     0.6612
dissimilar, expect low                  triangle A  (0-1-2)       star S5                     0.3902
1-WL / GIN blind spot                   two triangles (2 x C3)    6-cycle A                   1.0000


## 5. Full similarity matrix

Pairwise cosine over a small zoo of graphs. The diagonal is $1.0$ (a graph vs itself), and the matrix is symmetric.

In [6]:
zoo = [triangle_a, c6_a, two_triangles, p6, s5, complete(4, "K4"), star(3, "star S3")]

print("Similarity matrix (cosine, all in [0, 1]):\n")
header = "".join(f"{i:>8}" for i in range(len(zoo)))
print(f"{'':>26}{header}")
for i, gi in enumerate(zoo):
    row = "".join(f"{score(gi, gj):>8.3f}" for gj in zoo)
    print(f"{i}: {gi.name:<23}{row}")

Similarity matrix (cosine, all in [0, 1]):

                                 0       1       2       3       4       5       6
0: triangle A  (0-1-2)       1.000   1.000   1.000   0.930   0.390   0.444   0.421
1: 6-cycle A                 1.000   1.000   1.000   0.930   0.390   0.444   0.421
2: two triangles (2 x C3)    1.000   1.000   1.000   0.930   0.390   0.444   0.421
3: path P6                   0.930   0.930   0.930   1.000   0.661   0.524   0.683
4: star S5                   0.390   0.390   0.390   0.661   1.000   0.465   0.908
5: K4                        0.444   0.444   0.444   0.524   0.465   1.000   0.664
6: star S3                   0.421   0.421   0.421   0.683   0.908   0.664   1.000
